In [ ]:
# Install dependencies (uncomment if running in Colab or a fresh environment)
!pip install datasets[audio] transformers accelerate evaluate jiwer tensorboard

## Whisper Custom Dataset Fine-Tuning (with Custom Tokenizer)

This notebook demonstrates how to fine-tune Whisper on a custom dataset using a custom tokenizer.
- No config files or CLI arguments: all paths and parameters are hardcoded for clarity.
- Assumes your custom tokenizer is in './manipuri_tokenizer/'
- Assumes your custom dataset TSV is './datasets/lamzing/data.tsv' and audio files are in './datasets/lamzing/audio/'

In [ ]:
import os
import pandas as pd
import torch
import numpy as np
from datasets import Dataset, DatasetDict, Audio
from transformers import (
    WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor,
    WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
)
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

## 1. Load Custom Dataset
- The TSV file should have columns: 'path' (audio filename) and 'sentence' (transcription)
- Audio files should be in the 'audio' directory next to the TSV file

In [ ]:
custom_dataset_path = './datasets/lamzing/data.tsv'
audio_dir = os.path.join(os.path.dirname(custom_dataset_path), 'audio')
df = pd.read_csv(custom_dataset_path, sep='\t')
df['path'] = df['path'].apply(lambda p: os.path.join(audio_dir, p))
df = df.dropna(subset=['path', 'sentence'])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
n_train = int(0.9 * len(df))
df_train = df.iloc[:n_train]
df_test = df.iloc[n_train:]
ds_train = Dataset.from_pandas(df_train, preserve_index=False)
ds_test = Dataset.from_pandas(df_test, preserve_index=False)
dataset = DatasetDict({'train': ds_train, 'test': ds_test})
dataset = dataset.cast_column('path', Audio(sampling_rate=16000))

## 2. Load Whisper Model, Feature Extractor, and Custom Tokenizer
- The custom tokenizer directory must contain all tokenizer files
- The feature extractor and processor are loaded from the base Whisper model

In [ ]:
whisper_pretrained = 'openai/whisper-small'  # or your base model
custom_tokenizer_dir = './manipuri_tokenizer'
feature_extractor = WhisperFeatureExtractor.from_pretrained(whisper_pretrained)
tokenizer = WhisperTokenizer.from_pretrained(custom_tokenizer_dir, task='transcribe')
processor = WhisperProcessor.from_pretrained(whisper_pretrained)
model = WhisperForConditionalGeneration.from_pretrained(whisper_pretrained)
model.generation_config.language = None  # Monolingual
model.generation_config.task = 'transcribe'
model.config.use_cache = False

## 3. Prepare Dataset for Whisper


In [ ]:
def prepare_dataset(batch):
    audio = batch['path']
    batch['input_features'] = feature_extractor(audio['array'], sampling_rate=audio['sampling_rate']).input_features[0]
    batch['labels'] = tokenizer(batch['sentence']).input_ids
    return batch

dataset = dataset.map(prepare_dataset, remove_columns=dataset['train'].column_names, num_proc=1)

## 4. Data Collator

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        if "attention_mask" not in batch:
            batch["attention_mask"] = torch.ones(batch["input_features"].shape[:-1], dtype=torch.long)
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

## 5. Training Arguments

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir='./checkpoints/lamzing-whisper-small-mni',
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy='steps',
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
    save_total_limit=3,
)

## 6. Metrics

In [ ]:
metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

## 7. Trainer and Training

In [ ]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)
trainer.train()

## 8. Save Model and Tokenizer

In [ ]:
processor.save_pretrained(training_args.output_dir)
feature_extractor.save_pretrained(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)